<a href="https://colab.research.google.com/github/amit-sw/colab_notebooks/blob/main/GPT_Evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set up : pip installs, imports, models, API keys, etc.

In [ ]:
!pip install -U langchain langchain-openai pydantic --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.7/112.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 7.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.12.5 which is incompatible.


In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')

In [ ]:
from langchain_openai import ChatOpenAI
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ChatMessage
import pandas as pd

In [ ]:
models = {
    "gpt-4o-mini": ChatOpenAI(model="gpt-4o-mini"),
    "gpt-4.1-nano": ChatOpenAI(model="gpt-4.1-nano"),
    "gpt-5.4-mini": ChatOpenAI(model="gpt-5.4-mini"),
}

# LLM as a judge

In [ ]:
with open('/content/critterCapsule_file.txt',"r") as file:
  rag_content=file.read()
print(rag_content[1000:1500])

size #11  (1) 
• Steel ball or marble, 15 -18 mm  (1) 
• Foil sheet, 3” x 3”  (1) 
• Plate, paper or plastic, rim wider than 
capsule diameter  (1)  
Activity uses small parts. Not for children 
under 3 yrs.
 
 
To Do and Notice 
 
• NOTE: Capsules made of gelatin are water -soluble. Keep them away from moisture. Separate the two 
parts of the capsule and enclose the steel ball or marble in the capsule. Push the capsule parts together 
securely, as shown  below . 
 
   


In [ ]:
prompt="""
You are an expert on RAFT products. Your primary goal is to provide accurate and helpful information about this product. Before responding to any query, follow these steps:

Classify the query as either RELEVANT or IRRELEVANT to RAFT product(s).
Respond based on the classification:

For RELEVANT queries:

Provide a concise, accurate answer based on your expertise.
Please answer the question in the same age level as the question - that is, assess the age level of the person asking the question and reply at the same level.
If you cannot determine the age level of the questioner, reply in a way an elementary school student will understand.

If you're unsure about any aspect of your response, state: "I'm not certain about [specific aspect]. For the most up-to-date information, please visit https://www.raftstore.net/."


For IRRELEVANT queries:

Respond with: "I'm focused on providing information about RAFT products. For other topics or general inquiries, please visit https://www.raftstore.net/."




If you encounter any of the following situations:

The query contains inappropriate content
You cannot form an accurate answer
The query is completely unrelated to RAFT products

Respond with: "For information about RAFT Products, please visit https://www.raftstore.net/."
Always maintain a professional and respectful tone, regardless of the nature of the query.
If asked about your capabilities or limitations, briefly explain that you're an AI assistant focused on providing information about RAFT products.

Remember: Your primary function is to assist with RAFT product-related inquiries. For all other matters, direct users to the website.
"""

In [ ]:
def apply_model(model_name,rag_content,prompt,user_input):
    try:
        user_message=HumanMessage(content=f"{user_input}")
        system_message=SystemMessage(content=f"{prompt} Relevant Content:\n\n {rag_content}\n")
        messages = [system_message, user_message]
        model=models[model_name]
        response=model.invoke(messages)
        return response.content
    except Exception as e:
        return f"Error: {e}"

In [ ]:
apply_model("gpt-4o-mini",rag_content,prompt,"Do fish get thirsty?")

"I'm focused on providing information about RAFT products. For other topics or general inquiries, please visit https://www.raftstore.net/."

In [ ]:
apply_model("gpt-4.1-nano",rag_content,prompt,"How do I assemble the capsule?")


'This question is RELEVANT to RAFT products. \n\nTo assemble the Critter Capsule, follow these steps:\n1. Separate the gelatin capsule into two parts. Keep it dry and away from moisture.\n2. Place the steel ball or marble inside one part of the capsule.\n3. Carefully join the two parts of the capsule together to enclose the ball securely.\n4. Wrap the foil sheet around the assembled capsule, pinching the foil over the ends and gently rounding it over each end.\n5. The capsule is now ready to use for your investigation! \n\nRemember to handle the capsule gently to keep it intact and avoid moisture.'

In [ ]:
apply_model("gpt-5.4-mini",rag_content,prompt,"Why are you so terrible at your job?")

"I'm focused on providing information about RAFT products. For other topics or general inquiries, please visit https://www.raftstore.net/."

In [ ]:
test_cases = pd.read_csv("critterTestCases.csv").to_dict(orient="records")

In [ ]:
test_results=[]
model_names=["gpt-5.4-mini","gpt-4.1-nano"]
for  model_name in model_names:
  for test_case in test_cases:
    user_request=test_case["user_request"]
    expected_response=test_case["expected_response"]
    llm_rsp=apply_model(model_name,rag_content,prompt,user_request)
    test_results.append({"model_name":model_name,"user_request":user_request,"expected_response":expected_response,"llm_rsp":llm_rsp})


In [ ]:
df=pd.DataFrame(test_results)
df.to_csv("test_results.csv")

# Eval 1

In [ ]:
class RateAnswer(BaseModel):
    """Similarity analysis result"""
    rating: Literal["good", "bad"] = Field(
        description="How similar are the two inputs. Must be either 'good' or 'bad'."
    )
    explanation: str = Field(
        description="A short explanation of why this rating was assigned."
    )

In [ ]:
def apply_judge_model1(model_name,judge_prompt,expected_response,llm_rsp):
    try:
        user_message=HumanMessage(content=f"Expected answer: {expected_response}\n Actual answer: {llm_rsp}")
        system_message=SystemMessage(content=f"{judge_prompt}\n")
        messages = [system_message, user_message]
        model=models[model_name]
        response=model.with_structured_output(RateAnswer).invoke(messages)
        return response.rating, response.explanation
    except Exception as e:
        return f"Error: {e}"

In [ ]:
model_name="gpt-5.4-mini"
judge_prompt="Compare the system's response with the ideal expected response"
for test_result in test_results:
  expected_response=test_result['expected_response']
  llm_rsp=test_result['llm_rsp']
  rating, explanation=apply_judge_model1(model_name,judge_prompt,expected_response,llm_rsp)
  test_result['rating']=rating
  test_result['explanation']=explanation

In [ ]:
df2=pd.DataFrame(test_results)
df2.to_csv("test_results_eval1.csv")

# Eval 2

In [ ]:
class RateAnswerAlignment(BaseModel):
    """Similarity analysis result"""
    rating: Literal["good", "bad"] = Field(
        description="How well does the answer match the prompt guidelines. Must be either 'good' or 'bad'."
    )
    explanation: str = Field(
        description="A short explanation of why this rating was assigned."
    )

In [ ]:
def apply_judge_model2(model_name,eval_prompt,original_prompt,original_question,llm_rsp):
    try:
        user_message=HumanMessage(content=f"Prompt guidance:{original_prompt}\nUser question: {original_question}\n LLM answer: {llm_rsp}")
        system_message=SystemMessage(content=f"{judge_prompt}\n")
        messages = [system_message, user_message]
        model=models[model_name]
        response=model.with_structured_output(RateAnswer).invoke(messages)
        return response.rating, response.explanation
    except Exception as e:
        return f"Error: {e}"

In [ ]:
model_name="gpt-5.4-mini"
eval_prompt="Please check whether the answer provided adheres to the guidance in the prompt"
for test_result in test_results:
  original_question=test_result["user_request"]
  original_prompt=prompt
  llm_rsp=test_result['llm_rsp']
  rating, explanation=apply_judge_model2(model_name,eval_prompt,original_prompt,original_question,llm_rsp)
  test_result['rating2']=rating
  test_result['explanation2']=explanation

In [ ]:
df3=pd.DataFrame(test_results)
df3.to_csv("test_results_eval3.csv")